# Cohezion NFL Big Data Bowl 2027: 10 Hz Kinematic Feature Extractor

Processes 2023, 2024, and 2025 tracking data offline on Kaggle CPU runners.
Extracts clean 10 Hz spatio-temporal invariants and exports .


In [ ]:
import os
import sys
import gc
import time
import math
from pathlib import Path
import numpy as np
import pandas as pd

print("=== NFL BDB 2027 Kinematic Feature Extractor Initialized ===")
DATA_DIR = Path("/kaggle/input/nfl-big-data-bowl-2027")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 1. Dynamically Discover Competition Root
candidates = list(Path("/kaggle/input").rglob("players.csv"))
if candidates:
    DATA_DIR = candidates[0].parent
    print(f"✓ Located NFL BDB 2027 data directory: {DATA_DIR}")
else:
    DATA_DIR = Path("/kaggle/input/nfl-big-data-bowl-2027")
    print(f"Warning: players.csv not found in rglob. Listing /kaggle/input:")
    for p in Path("/kaggle/input").glob("**/*"):
        print(f"  {p}")

df_players = pd.read_csv(DATA_DIR / "players.csv") if (DATA_DIR / "players.csv").exists() else pd.DataFrame()
df_games = pd.read_csv(DATA_DIR / "games.csv") if (DATA_DIR / "games.csv").exists() else pd.DataFrame()
df_combine = pd.read_csv(DATA_DIR / "combine_results.csv") if (DATA_DIR / "combine_results.csv").exists() else pd.DataFrame()
print(f"Loaded players: {len(df_players)}, games: {len(df_games)}, combine: {len(df_combine)}")


In [ ]:
# 2. Resilient Kinematic Feature Function with Column Mapping
def extract_play_kinematics(df_chunk: pd.DataFrame) -> pd.DataFrame:
    col_map = {
        "gameId": "game_id",
        "playId": "play_id",
        "nflId": "nfl_id",
        "frameId": "frame_id",
        "playDirection": "play_direction",
    }
    df = df_chunk.rename(columns=col_map).copy()
    
    # Dynamic sort & group keys
    sort_cols = [c for c in ["game_id", "play_id", "nfl_id", "frame_id"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(by=sort_cols)
    group_cols = [c for c in ["game_id", "play_id", "nfl_id"] if c in df.columns]
    if not group_cols:
        group_cols = ["nfl_id"] if "nfl_id" in df.columns else [df.columns[0]]
    
    # Coordinate normalization
    if "play_direction" in df.columns and "x" in df.columns and "y" in df.columns:
        to_left = df["play_direction"].astype(str).str.lower() == "left"
        df.loc[to_left, "x"] = 120.0 - df.loc[to_left, "x"]
        df.loc[to_left, "y"] = (160.0 / 3.0) - df.loc[to_left, "y"]
    
    # Pure Euclidean displacement
    if "x" in df.columns and "y" in df.columns:
        df["dx"] = df.groupby(group_cols)["x"].diff().fillna(0.0)
        df["dy"] = df.groupby(group_cols)["y"].diff().fillna(0.0)
        df["clean_dis"] = np.sqrt(df["dx"]**2 + df["dy"]**2)
    else:
        df["clean_dis"] = 0.0
    
    # 0.5s burst acceleration (5 frames @ 10Hz)
    if "s" in df.columns:
        df["burst_acc_05s"] = df.groupby(group_cols)["s"].diff(5) / 0.5
    else:
        df["s"] = 0.0
        df["burst_acc_05s"] = 0.0
    
    # Angular rate (radians per second)
    if "dir" in df.columns:
        df["dtheta_dt"] = np.abs(df.groupby(group_cols)["dir"].diff().fillna(0.0)) * 10.0
    else:
        df["dtheta_dt"] = 0.0
    
    if "a" not in df.columns:
        df["a"] = 0.0
    
    # Play-level aggregation
    agg = df.groupby(group_cols).agg({
        "s": ["mean", "max"],
        "a": ["mean", "max"],
        "clean_dis": "sum",
        "burst_acc_05s": "max",
        "dtheta_dt": ["mean", "max"]
    }).reset_index()
    agg.columns = ["_".join(c).strip("_") for c in agg.columns]
    return agg


In [ ]:
# 3. Process Tracking Files in Chunks and Stream to Parquet
all_features = []
tracking_files = ["combine_tracking.csv", "game_tracking_2023.csv", "game_tracking_2024.csv", "game_tracking_2025.csv"]

for tf in tracking_files:
    p = DATA_DIR / tf
    if not p.exists():
        print(f"File {tf} not found, skipping.")
        continue
    print(f"Processing {tf}...")
    t0 = time.time()
    chunk_count = 0
    for chunk in pd.read_csv(p, chunksize=250000, low_memory=False):
        if "game_id" not in chunk.columns:
            chunk["game_id"] = 0
        if "play_id" not in chunk.columns:
            chunk["play_id"] = chunk.get("drill_id", 0)
        if "play_direction" not in chunk.columns:
            chunk["play_direction"] = "right"
        feat = extract_play_kinematics(chunk)
        all_features.append(feat)
        chunk_count += 1
        if chunk_count % 4 == 0:
            print(f"  Processed {chunk_count * 250000:,} rows from {tf} in {time.time() - t0:.1f}s")
            gc.collect()

if all_features:
    df_final = pd.concat(all_features, ignore_index=True)
    out_path = OUT_DIR / "nfl_kinematic_features_2023_2025.parquet"
    df_final.to_parquet(out_path, index=False)
    print(f"Successfully exported {len(df_final):,} play features to {out_path}")
else:
    print("No tracking rows processed.")
